# Sarvam-1 Evaluation on Odia GSM8K

Evaluates `ai4bharat/sarvam-1` on the `tripathysagar/odia-gsm8k` dataset.

**Metrics tracked:**
- **Accuracy**: exact-match on the final numerical answer
- **Latency**: TTFT, total generation time, tokens/sec (throughput), time per output token (TPOT), end-to-end latency

## 1. Setup

In [ ]:
# Install / upgrade dependencies
# %pip install -q transformers datasets accelerate torch python-dotenv pandas matplotlib seaborn tqdm

In [ ]:
import os
import re
import json
import time
import threading
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

load_dotenv()

sns.set_theme(style="whitegrid", palette="muted")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
HF_TOKEN        = os.getenv("HF_TOKEN")
MODEL_ID        = os.getenv("MODEL_ID",        "ai4bharat/sarvam-1")
DATASET_ID      = os.getenv("DATASET_ID",      "tripathysagar/odia-gsm8k")
DATASET_SPLIT   = os.getenv("DATASET_SPLIT",   "test")
FEW_SHOT_SPLIT  = os.getenv("FEW_SHOT_SPLIT",  "train")
NUM_FEW_SHOT    = int(os.getenv("NUM_FEW_SHOT",    "3"))
QUESTION_COL    = os.getenv("QUESTION_COL",    "question")
ANSWER_COL      = os.getenv("ANSWER_COL",      "answer")
MAX_NEW_TOKENS  = int(os.getenv("MAX_NEW_TOKENS",  "512"))
NUM_EVAL_SAMPLES= int(os.getenv("NUM_EVAL_SAMPLES", "100"))
DEVICE          = os.getenv("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR     = Path(os.getenv("RESULTS_DIR", "results"))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model        : {MODEL_ID}")
print(f"Dataset      : {DATASET_ID}  (eval={DATASET_SPLIT}, shots={FEW_SHOT_SPLIT})")
print(f"Few-shot     : {NUM_FEW_SHOT}")
print(f"Question col : {QUESTION_COL}")
print(f"Answer col   : {ANSWER_COL}")
print(f"Device       : {DEVICE}")
print(f"Samples      : {NUM_EVAL_SAMPLES if NUM_EVAL_SAMPLES > 0 else 'all'}")

## 2b. Experiment Tracking (Comet ML)

Logs each eval run as a Comet experiment in `COMET_EVAL_PROJECT`. Tag with `EVAL_RUN_TAG` (`base` / `sft` / `grpo`) — or leave blank and it's inferred from `MODEL_ID`. Open the project in Comet → "Compare" view to see all three runs side-by-side.

If `COMET_API_KEY` is not set, this is a no-op and the rest of the notebook still works.

In [ ]:
COMET_API_KEY      = os.getenv("COMET_API_KEY")
COMET_WORKSPACE    = os.getenv("COMET_WORKSPACE")
COMET_EVAL_PROJECT = os.getenv("COMET_EVAL_PROJECT", "sarvam1-odia-gsm8k-eval")
EVAL_RUN_TAG       = os.getenv("EVAL_RUN_TAG", "").strip().lower()

# Infer run tag from model id if not set explicitly
if not EVAL_RUN_TAG:
    mid = MODEL_ID.lower()
    if   "grpo" in mid: EVAL_RUN_TAG = "grpo"
    elif "sft"  in mid: EVAL_RUN_TAG = "sft"
    else:               EVAL_RUN_TAG = "base"

experiment = None
if COMET_API_KEY:
    try:
        from comet_ml import Experiment
        experiment = Experiment(
            api_key=COMET_API_KEY,
            workspace=COMET_WORKSPACE or None,
            project_name=COMET_EVAL_PROJECT,
            log_code=False,
            log_graph=False,
            auto_param_logging=False,
            auto_metric_logging=False,
        )
        experiment.set_name(f"{EVAL_RUN_TAG}-{MODEL_ID.split('/')[-1]}")
        experiment.add_tags([EVAL_RUN_TAG, "eval", DATASET_SPLIT])
        experiment.log_parameters({
            "model_id":         MODEL_ID,
            "dataset_id":       DATASET_ID,
            "dataset_split":    DATASET_SPLIT,
            "num_few_shot":     NUM_FEW_SHOT,
            "max_new_tokens":   MAX_NEW_TOKENS,
            "num_eval_samples": NUM_EVAL_SAMPLES,
            "device":           DEVICE,
            "run_tag":          EVAL_RUN_TAG,
        })
        print(f"Comet experiment started — project={COMET_EVAL_PROJECT}, tag={EVAL_RUN_TAG}")
    except Exception as exc:
        print(f"Comet init failed (continuing without tracking): {exc}")
        experiment = None
else:
    print(f"COMET_API_KEY not set — eval will run but won't be logged to Comet. (Run tag would be: {EVAL_RUN_TAG})")

## 3. Load Dataset

In [ ]:
dataset = load_dataset(DATASET_ID, token=HF_TOKEN)
print(dataset)

eval_data = dataset[DATASET_SPLIT]

if NUM_EVAL_SAMPLES > 0:
    eval_data = eval_data.select(range(min(NUM_EVAL_SAMPLES, len(eval_data))))

print(f"\nUsing split '{DATASET_SPLIT}' — {len(eval_data)} samples")
print("\nSample entry (verify gold-answer format below):")
print(f"Question: {eval_data[0][QUESTION_COL]}")
print(f"Answer  : {eval_data[0][ANSWER_COL]!r}")

In [ ]:
print("Available columns:", eval_data.column_names)
print(f"Question column  : {QUESTION_COL}")
print(f"Answer column    : {ANSWER_COL}")

assert QUESTION_COL in eval_data.column_names, f"QUESTION_COL='{QUESTION_COL}' not found in {eval_data.column_names}"
assert ANSWER_COL   in eval_data.column_names, f"ANSWER_COL='{ANSWER_COL}' not found in {eval_data.column_names}"

In [ ]:
# Build few-shot examples from a separate split (Sarvam-1 is a base model — few-shot helps a lot)
FEW_SHOT_EXAMPLES = []
if NUM_FEW_SHOT > 0:
    shots = dataset[FEW_SHOT_SPLIT].select(range(NUM_FEW_SHOT))
    FEW_SHOT_EXAMPLES = [
        {"question": s[QUESTION_COL], "answer": str(s[ANSWER_COL])}
        for s in shots
    ]

print(f"Loaded {len(FEW_SHOT_EXAMPLES)} few-shot examples from split '{FEW_SHOT_SPLIT}'")
if FEW_SHOT_EXAMPLES:
    print("\nFirst few-shot example:")
    print(f"Q: {FEW_SHOT_EXAMPLES[0]['question'][:150]}...")
    print(f"A: {FEW_SHOT_EXAMPLES[0]['answer'][:200]}...")

## 4. Load Model & Tokenizer

In [ ]:
print(f"Loading tokenizer from {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model from {MODEL_ID} ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else DEVICE,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

## 5. Helper Functions

In [ ]:
SYSTEM_PROMPT = (
    "ଆପଣ ଜଣେ ସହାୟକ ଗଣିତ ସହକାରୀ ଅଟନ୍ତି। "
    "ତଳେ ଦିଆଯାଇଥିବା ସମସ୍ୟାକୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ସମାଧାନ କରନ୍ତୁ। "
    "ଶେଷରେ, ଆପଣଙ୍କର ଚୂଡ଼ାନ୍ତ ସାଂଖ୍ୟିକ ଉତ୍ତରକୁ ଏକ ନୂଆ ଧାଡ଼ିରେ '####' ସହିତ ଆରମ୍ଭ କରି ଲେଖନ୍ତୁ।"
)

# Odia digits (୦-୯) → Arabic digits (0-9). Sarvam-1 may emit numerals in Odia script.
ODIA_TO_ARABIC = str.maketrans("୦୧୨୩୪୫୬୭୮୯", "0123456789")


def normalize_digits(text: str) -> str:
    return text.translate(ODIA_TO_ARABIC)


def build_prompt(question: str) -> str:
    """Format question with optional few-shot context.
    Works for base (few-shot) and SFT/GRPO (set NUM_FEW_SHOT=0) models —
    the SFT/GRPO format matches the zero-shot fallback exactly."""
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user",      "content": ex["question"]})
            messages.append({"role": "assistant", "content": ex["answer"]})
        messages.append({"role": "user", "content": question})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Base-model / SFT / GRPO fallback: plain text format
    parts = [SYSTEM_PROMPT, ""]
    for ex in FEW_SHOT_EXAMPLES:
        parts.append(f"ପ୍ରଶ୍ନ: {ex['question']}")
        parts.append(f"ଉତ୍ତର: {ex['answer']}")
        parts.append("")
    parts.append(f"ପ୍ରଶ୍ନ: {question}")
    parts.append("ଉତ୍ତର:")
    return "\n".join(parts)


def extract_numerical_answer(text: str) -> Optional[float]:
    """Extract the final numerical answer; handles both Arabic and Odia digits."""
    text = normalize_digits(text)
    # GSM8K convention: #### <number>
    marker_match = re.search(r"####\s*([\-\d,\.]+)", text)
    if marker_match:
        num_str = marker_match.group(1).replace(",", "").rstrip(".")
        try:
            return float(num_str)
        except ValueError:
            pass
    # Fallback: last standalone number in text
    numbers = re.findall(r"-?\d+(?:[,\.]\d+)*", text)
    if numbers:
        try:
            return float(numbers[-1].replace(",", "").rstrip("."))
        except ValueError:
            pass
    return None


def answers_match(pred: Optional[float], gold: Optional[float], tol: float = 1e-3) -> bool:
    if pred is None or gold is None:
        return False
    return abs(pred - gold) <= tol * max(1.0, abs(gold))


# --- Correctness side-signals: format + language adherence ---

_FORMAT_RE = re.compile(r"####\s*-?\d")


def has_format_marker(text: str) -> bool:
    """True iff output contains '#### <number>' (Arabic or Odia digits) — the GSM8K answer convention."""
    return bool(_FORMAT_RE.search(normalize_digits(text)))


def odia_script_ratio(text: str) -> float:
    """Fraction of *script* characters in Odia (vs Latin). Ignores digits, punctuation, whitespace.
    Higher = better Odia adherence; low values mean the model code-switched to English."""
    odia  = sum(1 for c in text if "\u0B00" <= c <= "\u0B7F" and c.isalpha())
    latin = sum(1 for c in text if c.isascii() and c.isalpha())
    total = odia + latin
    return odia / total if total > 0 else 0.0

In [ ]:
@dataclass
class LatencyRecord:
    prompt_tokens: int = 0
    output_tokens: int = 0
    ttft_ms: float = 0.0          # time to first token (chunk)
    total_latency_ms: float = 0.0 # wall-clock end-to-end
    throughput_tok_per_sec: float = 0.0
    tpot_ms: float = 0.0          # avg time per generated token (excluding first)


def _cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def generate_with_metrics(prompt: str) -> tuple[str, LatencyRecord]:
    """Greedy decode + latency stats. Token count comes from generate(), not text re-encoding."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_tokens = inputs["input_ids"].shape[-1]

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        streamer=streamer,
    )

    gen_out: dict = {}

    def _run_generation():
        with torch.no_grad():
            gen_out["result"] = model.generate(**gen_kwargs)

    _cuda_sync()
    t_start = time.perf_counter()

    thread = threading.Thread(target=_run_generation)
    thread.start()

    generated_chunks: list[str] = []
    ttft_ms: Optional[float] = None
    for token_text in streamer:
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t_start) * 1000
        generated_chunks.append(token_text)

    thread.join()
    _cuda_sync()
    total_latency_ms = (time.perf_counter() - t_start) * 1000

    full_output = "".join(generated_chunks)

    # Accurate output-token count from generate(), not text re-encoding
    result = gen_out.get("result")
    if result is not None:
        output_tokens = int(result.sequences.shape[-1] - prompt_tokens)
    else:
        output_tokens = len(tokenizer.encode(full_output, add_special_tokens=False))

    if ttft_ms is None:
        ttft_ms = total_latency_ms

    # TPOT = inter-token latency = (total - ttft) / (output_tokens - 1)
    if output_tokens > 1:
        tpot_ms = (total_latency_ms - ttft_ms) / (output_tokens - 1)
    else:
        tpot_ms = total_latency_ms

    throughput = output_tokens / (total_latency_ms / 1000) if total_latency_ms > 0 else 0.0

    return full_output, LatencyRecord(
        prompt_tokens=prompt_tokens,
        output_tokens=output_tokens,
        ttft_ms=ttft_ms,
        total_latency_ms=total_latency_ms,
        throughput_tok_per_sec=throughput,
        tpot_ms=tpot_ms,
    )

In [ ]:
results = []

for idx, sample in enumerate(tqdm(eval_data, desc="Evaluating")):
    question = sample[QUESTION_COL]
    gold_raw = sample[ANSWER_COL]
    gold_num = extract_numerical_answer(str(gold_raw))

    prompt = build_prompt(question)

    try:
        output_text, latency = generate_with_metrics(prompt)
    except Exception as exc:
        print(f"[{idx}] Generation error: {exc}")
        output_text = ""
        latency = LatencyRecord()

    pred_num   = extract_numerical_answer(output_text)
    correct    = answers_match(pred_num, gold_num)
    format_ok  = has_format_marker(output_text)
    odia_ratio = odia_script_ratio(output_text)

    results.append({
        "idx":               idx,
        "question":          question,
        "gold_raw":          str(gold_raw),
        "gold_num":          gold_num,
        "output_text":       output_text,
        "pred_num":          pred_num,
        "correct":           correct,
        "format_ok":         format_ok,
        "odia_ratio":        odia_ratio,
        "prompt_tokens":     latency.prompt_tokens,
        "output_tokens":     latency.output_tokens,
        "ttft_ms":           latency.ttft_ms,
        "total_latency_ms":  latency.total_latency_ms,
        "tpot_ms":           latency.tpot_ms,
        "throughput_tok_s":  latency.throughput_tok_per_sec,
    })

df = pd.DataFrame(results)
print(f"\nEvaluation complete — {len(df)} samples")

In [ ]:
# Warmup: amortize CUDA init / kernel compilation so the first eval sample isn't an outlier
print("Warming up generation...")
_warm_prompt = build_prompt(eval_data[0][QUESTION_COL])
_, _warm_lat = generate_with_metrics(_warm_prompt)
print(f"Warmup done — {_warm_lat.total_latency_ms:.0f} ms, {_warm_lat.output_tokens} tokens")

In [ ]:
total   = len(df)
correct = int(df["correct"].sum())
no_pred = int(df["pred_num"].isna().sum())
accuracy = correct / total * 100

# Side-signals
format_ok_pct       = df["format_ok"].mean() * 100
odia_ratio_mean     = df["odia_ratio"].mean() * 100
odia_pure_pct       = (df["odia_ratio"] >= 0.95).mean() * 100   # ≥95% Odia script
english_drift_pct   = (df["odia_ratio"] <  0.50).mean() * 100   # majority Latin = drift

# Decompose accuracy: how much correctness is hidden behind format/language failures?
acc_among_formatted = df.loc[df["format_ok"], "correct"].mean() * 100 if df["format_ok"].any() else 0.0

print("=" * 48)
print(f"Total samples         : {total}")
print(f"Accuracy (final ans)  : {accuracy:.2f}%   ({correct}/{total})")
print(f"  └─ among format-ok  : {acc_among_formatted:.2f}%")
print(f"No answer found       : {no_pred} ({no_pred/total*100:.1f}%)")
print("-" * 48)
print(f"Format adherence      : {format_ok_pct:.2f}%   (output contains '#### N')")
print(f"Odia script ratio     : {odia_ratio_mean:.2f}%  (mean over outputs)")
print(f"  ≥95% Odia (pure)    : {odia_pure_pct:.2f}%")
print(f"  <50% Odia (drifted) : {english_drift_pct:.2f}%")
print("=" * 48)

## 7. Accuracy Metrics

In [ ]:
total   = len(df)
correct = df["correct"].sum()
no_pred = df["pred_num"].isna().sum()
accuracy = correct / total * 100

print("=" * 40)
print(f"Total samples   : {total}")
print(f"Correct         : {correct}")
print(f"Accuracy        : {accuracy:.2f}%")
print(f"No answer found : {no_pred} ({no_pred/total*100:.1f}%)")
print("=" * 40)

## 8. Latency Metrics

In [ ]:
latency_cols = ["ttft_ms", "total_latency_ms", "tpot_ms", "throughput_tok_s", "output_tokens"]
lat_summary  = df[latency_cols].describe(percentiles=[0.5, 0.75, 0.95, 0.99])

print("\n--- Latency Summary ---")
print(lat_summary.T[["mean", "50%", "75%", "95%", "99%", "min", "max"]].to_string())

## 9. Visualizations

In [ ]:
csv_path = RESULTS_DIR / "eval_results.csv"
df.to_csv(csv_path, index=False)
print(f"Raw results saved to {csv_path}")

summary = {
    "model":                MODEL_ID,
    "dataset":              DATASET_ID,
    "split":                DATASET_SPLIT,
    "num_few_shot":         NUM_FEW_SHOT,
    "num_samples":          total,
    "accuracy_pct":         round(accuracy, 4),
    "accuracy_among_formatted_pct": round(acc_among_formatted, 4),
    "no_answer_pct":        round(no_pred / total * 100, 4),
    "format_adherence_pct": round(format_ok_pct, 4),
    "odia_ratio_mean_pct":  round(odia_ratio_mean, 4),
    "odia_pure_pct":        round(odia_pure_pct, 4),
    "english_drift_pct":    round(english_drift_pct, 4),
    "latency": {
        "ttft_ms":          {p: round(np.percentile(df["ttft_ms"], p), 2) for p in [50, 95, 99]},
        "e2e_ms":           {p: round(np.percentile(df["total_latency_ms"], p), 2) for p in [50, 95, 99]},
        "tpot_ms":          {p: round(np.percentile(df["tpot_ms"], p), 2) for p in [50, 95, 99]},
        "throughput_tok_s": {"mean": round(df["throughput_tok_s"].mean(), 2),
                             "p50":  round(df["throughput_tok_s"].median(), 2)},
    },
}

summary_path = RESULTS_DIR / "eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))

# --- Push to Comet for cross-run comparison ---
if experiment is not None:
    # Flat metrics → Comet's compare view ranks/plots these across runs
    flat_metrics = {
        "accuracy_pct":                  summary["accuracy_pct"],
        "accuracy_among_formatted_pct":  summary["accuracy_among_formatted_pct"],
        "no_answer_pct":                 summary["no_answer_pct"],
        "format_adherence_pct":          summary["format_adherence_pct"],
        "odia_ratio_mean_pct":           summary["odia_ratio_mean_pct"],
        "odia_pure_pct":                 summary["odia_pure_pct"],
        "english_drift_pct":             summary["english_drift_pct"],
        "num_samples":                   summary["num_samples"],
        "ttft_ms_p50":                   summary["latency"]["ttft_ms"][50],
        "ttft_ms_p95":                   summary["latency"]["ttft_ms"][95],
        "ttft_ms_p99":                   summary["latency"]["ttft_ms"][99],
        "e2e_ms_p50":                    summary["latency"]["e2e_ms"][50],
        "e2e_ms_p95":                    summary["latency"]["e2e_ms"][95],
        "e2e_ms_p99":                    summary["latency"]["e2e_ms"][99],
        "tpot_ms_p50":                   summary["latency"]["tpot_ms"][50],
        "tpot_ms_p95":                   summary["latency"]["tpot_ms"][95],
        "throughput_tok_s_mean":         summary["latency"]["throughput_tok_s"]["mean"],
        "throughput_tok_s_p50":          summary["latency"]["throughput_tok_s"]["p50"],
    }
    experiment.log_metrics(flat_metrics)

    # Per-sample distributions (Comet renders histograms)
    for col in ["ttft_ms", "total_latency_ms", "tpot_ms", "throughput_tok_s", "output_tokens", "odia_ratio"]:
        experiment.log_histogram_3d(df[col].dropna().tolist(), name=col)

    # Assets: full results CSV, summary JSON, plot
    experiment.log_asset(str(csv_path),     file_name="eval_results.csv")
    experiment.log_asset(str(summary_path), file_name="eval_summary.json")
    plot_path = RESULTS_DIR / "eval_plots.png"
    if plot_path.exists():
        experiment.log_image(str(plot_path), name="eval_plots")

    # Per-sample table (Comet "Data" tab) — drop the long output_text to keep the table light
    table_df = df.drop(columns=["output_text"], errors="ignore").copy()
    experiment.log_table("per_sample_results.csv", tabular_data=table_df)

    experiment.end()
    print(f"\nLogged eval run to Comet — tag='{EVAL_RUN_TAG}', project='{COMET_EVAL_PROJECT}'")
    print("Open the project in Comet and use 'Compare' to overlay base/sft/grpo runs.")

## 10. Save Results

In [ ]:
csv_path = RESULTS_DIR / "eval_results.csv"
df.to_csv(csv_path, index=False)
print(f"Raw results saved to {csv_path}")

summary = {
    "model":                MODEL_ID,
    "dataset":              DATASET_ID,
    "split":                DATASET_SPLIT,
    "num_few_shot":         NUM_FEW_SHOT,
    "num_samples":          total,
    "accuracy_pct":         round(accuracy, 4),
    "no_answer_pct":        round(no_pred / total * 100, 4),
    "latency": {
        "ttft_ms":          {p: round(np.percentile(df["ttft_ms"], p), 2) for p in [50, 95, 99]},
        "e2e_ms":           {p: round(np.percentile(df["total_latency_ms"], p), 2) for p in [50, 95, 99]},
        "tpot_ms":          {p: round(np.percentile(df["tpot_ms"], p), 2) for p in [50, 95, 99]},
        "throughput_tok_s": {"mean": round(df["throughput_tok_s"].mean(), 2),
                             "p50":  round(df["throughput_tok_s"].median(), 2)},
    },
}

summary_path = RESULTS_DIR / "eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))

## 11. Error Analysis

In [ ]:
# Inspect a few incorrect predictions
errors = df[~df["correct"]].head(5)
for _, row in errors.iterrows():
    print(f"--- Sample {row['idx']} ---")
    print(f"Question : {row['question'][:120]}...")
    print(f"Gold     : {row['gold_num']}  (raw: {row['gold_raw'][:60]})")
    print(f"Predicted: {row['pred_num']}")
    print(f"Output   : {row['output_text'][:200]}...")
    print()